<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# Localhost and Service Binding

The previous notebook introduced TCP listeners and showed how to list them with `ss`. But a listening service may accept connections from the Duckiedrone itself while remaining unavailable from your base station.

Two details help explain this: where the requesting program runs, and which local addresses the service listens on.

In this notebook, you will interpret those addresses and start a temporary web server to observe a loopback-only connection.

## Identify the current network context

You have already used shells on the base station, inside the browser-based LX editor, and on physical and virtual Duckiedrones. The same address can mean different things in these different environments.

`localhost` refers back to the requesting program’s own network environment. Its loopback addresses are `127.0.0.1` for IPv4 and `::1` for IPv6.

For example, when running `dts code editor` in this LX root directory on your base station, a browser VS Code editor is opened. Its terminal, filesystem, processes, network interfaces, and `localhost` belong to a separate editor environment, not to the base station.

[Figure 1](#figure-1) shows where to open a terminal in the browser editor. Commands entered there run in the editor environment, not directly in the browser host environment.

<figure id="figure-1" class="lx-figure">
  <img src="../assets/images/vs-code-editor-terminal.jpg" alt="Browser editor interface with its terminal opened" style="display:block; width:80%; margin:0 auto;">
  <figcaption>Figure 1: Opening a terminal in the browser editor.</figcaption>
</figure>

Keep a separate base-station terminal open for `dts`, base-station credentials, and checks that need the base station's network path to a Duckiedrone. Each context has its own loopback address, as summarized in [Table 1](#table-1).

<table id="table-1" class="lx-table">
  <caption>Table 1: Network contexts and their <code>localhost</code> targets.</caption>
  <thead>
    <tr>
      <th>Command runs in</th>
      <th><code>localhost</code> normally refers to</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>dts code editor</code> terminal</td>
      <td>The editor environment</td>
    </tr>
    <tr>
      <td>Base-station terminal</td>
      <td>The base station</td>
    </tr>
    <tr>
      <td>Shell on a physical Duckiedrone</td>
      <td>That Duckiedrone</td>
    </tr>
    <tr>
      <td>Shell on a virtual Duckiedrone</td>
      <td>That Duckiedrone</td>
    </tr>
    <tr>
      <td>Service shell</td>
      <td>That service environment</td>
    </tr>
  </tbody>
</table>

After opening a shell on a Duckiedrone, subsequent commands run in that Duckiedrone environment. A service shell within a Duckiedrone is another separate context, so `localhost` there refers to that service environment rather than to the Duckiedrone, base station, or editor.

For example, `http://localhost:8080` is a Uniform Resource Locator (URL) for a web service. Opened in a base-station browser, it targets the base station; requested from a Duckiedrone shell, the same URL targets that Duckiedrone instead.

## Service binding

A service selects a protocol, port, and local address on which to accept connections. This is called __binding__.

Suppose a web service on a Duckiedrone listens at `127.0.0.1:8080`. A program in the same network environment can contact that listener. A request addressed to that Duckiedrone's Wi-Fi address, `192.168.1.201:8080`, does not match it.

Compare the possible listening addresses in [Table 2](#table-2).

<table id="table-2" class="lx-table">
  <caption>Table 2: How listening addresses affect reachability.</caption>
  <thead>
    <tr><th>Local listening address</th><th>Meaning</th></tr>
  </thead>
  <tbody>
    <tr><td><code>127.0.0.1:8080</code></td><td>IPv4 loopback only</td></tr>
    <tr><td><code>192.168.1.201:8080</code></td><td>The example Duckiedrone's Wi-Fi IPv4 address</td></tr>
    <tr><td><code>0.0.0.0:8080</code></td><td>All local IPv4 addresses in this network environment</td></tr>
    <tr><td><code>[::1]:8080</code></td><td>IPv6 loopback only</td></tr>
    <tr><td><code>[::]:8080</code></td><td>All local IPv6 addresses in this network environment</td></tr>
  </tbody>
</table>

Brackets distinguish the colons within an IPv6 address from the port separator.

`0.0.0.0` and `::` are __wildcard listening addresses__. Connect using the service’s actual hostname or address rather than copying a wildcard into your browser. Whether an IPv6 wildcard listener also accepts IPv4 depends on its configuration.

A suitable binding does not guarantee remote access: routing and access rules still apply. The Linux [`ip(7)` manual](https://man7.org/linux/man-pages/man7/ip.7.html) describes IPv4 binding in detail.

### Try it: start and inspect a local web server

Open __two terminals in the same Workspace or Ubuntu environment__. No Duckiedrone connection is needed.

In terminal 1, create an empty temporary directory and serve it over loopback:

```bash
SERVE_DIR=$(mktemp -d)
python3 -m http.server 8000 --bind 127.0.0.1 --directory "$SERVE_DIR"
```

`--directory` selects what the server serves; `--bind` selects its listening address. The command stays running until you stop it. See Python’s [`http.server` documentation](https://docs.python.org/3/library/http.server.html).

If port `8000` is already occupied, use `8001` consistently in this exercise rather than stopping the existing service.

In terminal 2, inspect the listener:

```bash
ss -ltn 'sport = :8000'
```

The filter selects local port `8000`. An illustrative result is:

```shell
State  Recv-Q Send-Q Local Address:Port Peer Address:Port
LISTEN 0      5      127.0.0.1:8000     0.0.0.0:*
```

Queue values may differ. The important field is __Local Address:Port__: the server listens on loopback.

Now send an HTTP request:

```bash
curl --head --max-time 5 http://127.0.0.1:8000
```

`--head` requests response headers without the page body; `--max-time 5` limits the operation. A successful response should begin with a line such as:

```shell
HTTP/1.0 200 OK
```

Terminal 1 should also log the request. You have observed both a TCP listener and an application responding through it.

In terminal 1, press __Ctrl-C__, then remove the empty temporary directory:

```bash
rmdir "$SERVE_DIR"
```

Repeat the `ss` command in terminal 2. The listener row should have disappeared.

__Question:__ Does the successful request prove that your base station’s browser can access a server running inside a separate Workspace network environment?

<details>
<summary>Reveal answer</summary>

No. Both commands used the Workspace’s loopback. The browser’s localhost belongs to the base station. Access between separate environments requires an appropriate connection path, potentially including forwarding.

</details>

### Recognize when forwarding is involved

Development tools sometimes make an internal service available through a forwarded port. [Figure 2](#figure-2) shows how a base-station browser reaches a service in a development environment.

<figure id="figure-2" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    base-station browser
      |
      v
    base-station address and port
      | forwarding
      v
    reachable service address and port
    inside the development environment
  </pre>
  <figcaption>
    Figure 2: Port forwarding from a base-station browser to a development service.
  </figcaption>
</figure>

__Port forwarding__ relays traffic between these endpoints. It requires a working path to the destination listener; it does not change the listener’s binding.

If a development tool lets your browser reach a Workspace service, check whether forwarding is configured before concluding that the two environments share `localhost`.

## Further reading

The Linux [network namespaces manual](https://man7.org/linux/man-pages/man7/network_namespaces.7.html) explains how network environments can be separated.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [1]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
